<center><h1>Le_Casey_HW3</h1></center>
<br>
<br>

Name: Casey Le
<br>
Github Username: parakeet89
<br>
USC ID: 8489799178

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.stats as stats

Get the AReM Data Set

In [2]:
columns = ['avg_rss12', 'var_rss12', 'avg_rss13', 'var_rss13', 'avg_rss23', 'var_rss23']
all_datasets = {}

for folder in sorted(os.listdir('../data/AReM')):
    folder_path = '../data/AReM/' + folder

    if os.path.isdir(folder_path):
        for file in sorted(os.listdir(folder_path)):

            if folder =='bending2' and file == 'dataset4.csv':
                df_prelim = pd.read_csv(folder_path + '/' + file, skiprows=5, header=None, names=['time'] + columns, sep=r'\s+')
                all_datasets[folder + '_' + file.replace('.csv', '')] = df_prelim

            elif file.endswith('.csv'):
                df_prelim = pd.read_csv(folder_path + '/' + file, skiprows=4, usecols=range(7))
                all_datasets[folder + '_' + file.replace('.csv', '')] = df_prelim

### (b) Test and Train Data

In [3]:
train_datasets = {}
test_datasets = {}

for dataset in all_datasets:
    if 'bending' in dataset:
        if dataset.endswith('dataset1') or dataset.endswith('dataset2'):
            test_datasets[dataset] = all_datasets[dataset]
        else:
            train_datasets[dataset] = all_datasets[dataset]
    else:
        if dataset.endswith('dataset1') or dataset.endswith('dataset2') or dataset.endswith('dataset3'):
            test_datasets[dataset] = all_datasets[dataset]
        else:
            train_datasets[dataset] = all_datasets[dataset]

### (c) Feature Extraction

#### i. Research

##### Time-domain features usually used in time series classification

* **Minimum**: Represents the lowest value of the time series x. This is the statistic I'd use to return the lowest value of a given predictor across the 2-minute span.
* **Maximum**: Represents the highest value of the time series x. This is the statistic I'd use to return the highest value of a given predictor across the 2-minute span.
* **Mean**: Returns the mean of x. This is the statistic I'd use to describe the average value of a predictor across the 2-minute span.
* **Standard deviation**: Returns the standard deviation of x. This is the statistic I'd use to describe how much the values of a given predictor vary around their mean across the 2-minute span.
* **Median**: Represents the middle value of the time series x when its values are sorted. This is the statistic I'd use to return the midpoint value of all the data points for a given predictor across the 2-minute span.
* **First quartile**: Similar to the median, except this statistic describes the value at the 25th percentile of a given predictor across the 2-minute span.
* **Third quartile**: Similar to the median, except this statistic describes the value at the 75th percentile of a given predictor across the 2-minute span.
* **Linear trend**: Represents a linear least-squares regression fitted to the values of the time series against their sequential positions. In other words, this feature describes the overall linear trend of the data across the 2-minute span (whether it's increasing, decreasing or relatively flat). The slope of the regression line indicates the direction and rate of change.

#### ii. Extraction

In [4]:
feature_datasets = {}

for dataset in all_datasets:
    df_prelim = all_datasets[dataset]
    dataset_dict = {}

    for column in columns:
        dataset_dict[column] = {}
        dataset_dict[column]['min'] = float(df_prelim[column].min())
        dataset_dict[column]['max'] = float(df_prelim[column].max())
        dataset_dict[column]['mean'] = float(df_prelim[column].mean())
        dataset_dict[column]['median'] = float(df_prelim[column].median())
        dataset_dict[column]['std'] = float(df_prelim[column].std())
        dataset_dict[column]['iqr_1'] = float(df_prelim[column].quantile(.25))
        dataset_dict[column]['iqr_3'] = float(df_prelim[column].quantile(.75))

    feature_datasets[dataset] = dataset_dict

df = pd.DataFrame.from_dict(
    {
        key: {
            (column, stat): value
            for column, stats in dataset.items()
            for stat, value in stats.items()
        }
        for key, dataset in feature_datasets.items()
    },
    orient='index'
)

df

avg_rss12                                             \
                        min    max       mean  median       std  iqr_1   
bending1_dataset1     37.25  45.00  40.624792  40.500  1.476967  39.25   
bending1_dataset2     38.00  45.67  42.812812  42.500  1.435550  42.00   
bending1_dataset3     35.00  47.40  43.954500  44.330  1.558835  43.00   
bending1_dataset4     33.00  47.75  42.179812  43.500  3.670666  39.15   
bending1_dataset5     33.00  45.75  41.678063  41.750  2.243490  41.33   
...                     ...    ...        ...     ...       ...    ...   
walking_dataset5      20.75  46.25  34.763333  35.290  4.742208  31.67   
walking_dataset6      21.50  51.00  34.935812  35.500  4.645944  32.00   
walking_dataset7      18.33  47.67  34.333042  34.750  4.948770  31.25   
walking_dataset8      18.33  45.75  34.599875  35.125  4.731790  31.50   
walking_dataset9      15.50  43.67  34.225875  34.750  4.441798  31.25   

                           var_rss12                   ... avg_rss23           \
                     iqr_3       min    max      mean  ...       std    iqr_1   
bending1_dataset1  42.0000       0.0   1.30  0.358604  ...  2.188449  33.0000   
bending1_dataset2  43.6700       0.0   1.22  0.372437  ...  1.995255  32.0000   
bending1_dataset3  45.0000       0.0   1.70  0.426250  ...  1.999604  35.3625   
bending1_dataset4  45.0000       0.0   3.00  0.696042  ...  3.849448  30.4575   
bending1_dataset5  42.7500       0.0   2.83  0.535979  ...  2.411026  28.4575   
...                    ...       ...    ...       ...  ...       ...      ...   
walking_dataset5   38.2500       0.0  12.68  4.223792  ...  3.174681  14.2500   
walking_dataset6   38.0625       0.0  12.21  4.115750  ...  3.192058  14.2375   
walking_dataset7   38.0000       0.0  12.48  4.396958  ...  3.000493  13.7500   
walking_dataset8   38.0000       0.0  15.37  4.398833  ...  2.905688  14.0000   
walking_dataset9   37.2500       0.0  17.24  4.354500  ...  2.992920  14.3300   

                         var_rss23                                          \
                   iqr_3       min    max      mean median       std iqr_1   
bending1_dataset1  36.00       0.0   1.92  0.570583  0.430  0.582915  0.00   
bending1_dataset2  34.50       0.0   3.11  0.571083  0.430  0.601010  0.00   
bending1_dataset3  36.50       0.0   1.79  0.493292  0.430  0.513506  0.00   
bending1_dataset4  36.33       0.0   2.18  0.613521  0.500  0.524317  0.00   
bending1_dataset5  31.25       0.0   1.79  0.383292  0.430  0.389164  0.00   
...                  ...       ...    ...       ...    ...       ...   ...   
walking_dataset5   18.33       0.0   9.39  3.288271  3.270  1.647528  2.05   
walking_dataset6   18.25       0.0  10.21  3.280021  3.015  1.700918  2.12   
walking_dataset7   18.00       0.0   8.01  3.261583  2.980  1.617290  2.05   
walking_dataset8   18.25       0.0   8.86  3.289542  3.015  1.680170  2.12   
walking_dataset9   18.25       0.0   9.42  3.479542  3.270  1.761146  2.24   

                           
                    iqr_3  
bending1_dataset1  1.3000  
bending1_dataset2  1.3000  
bending1_dataset3  0.9400  
bending1_dataset4  1.0000  
bending1_dataset5  0.5000  
...                   ...  
walking_dataset5   4.3050  
walking_dataset6   4.5000  
walking_dataset7   4.3200  
walking_dataset8   4.2600  
walking_dataset9   4.5375  

[88 rows x 42 columns]

#### iii. Standard Deviation

In [5]:
df_columns = list(df.columns)
df_columns

lst_std = {}
for column in df_columns:
    lst_std[column] = df[column].std()

print(pd.Series(lst_std))

avg_rss12  min       9.569975
           max       4.394362
           mean      5.335718
           median    5.440054
           std       1.772153
           iqr_1     6.153590
           iqr_3     5.138925
var_rss12  min       0.000000
           max       5.062729
           mean      1.574164
           median    1.412244
           std       0.884105
           iqr_1     0.946386
           iqr_3     2.125266
avg_rss13  min       2.956462
           max       4.875137
           mean      4.008380
           median    4.036396
           std       0.946710
           iqr_1     4.220658
           iqr_3     4.171628
var_rss13  min       0.000000
           max       2.183625
           mean      1.166114
           median    1.145586
           std       0.458242
           iqr_1     0.843620
           iqr_3     1.552504
avg_rss23  min       6.124001
           max       5.741238
           mean      5.675593
           median    5.813782
           std       1.024898
          

In [6]:
def std(x, axis):
    return np.std(x, axis=axis, ddof=1)

for column in df_columns:
    result = stats.bootstrap(data=(df[column],), statistic=std, confidence_level=.9, method='percentile')
    print(f"{column}: {result.confidence_interval}")

('avg_rss12', 'min'): ConfidenceInterval(low=np.float64(8.254647362450555), high=np.float64(10.765401508125068))
('avg_rss12', 'max'): ConfidenceInterval(low=np.float64(3.340056379898473), high=np.float64(5.2917041916824905))
('avg_rss12', 'mean'): ConfidenceInterval(low=np.float64(4.7064804927853965), high=np.float64(5.884544828175276))
('avg_rss12', 'median'): ConfidenceInterval(low=np.float64(4.801800931368866), high=np.float64(5.988713060917182))
('avg_rss12', 'std'): ConfidenceInterval(low=np.float64(1.5726204136792659), high=np.float64(1.9498843464771798))
('avg_rss12', 'iqr_1'): ConfidenceInterval(low=np.float64(5.564567795078061), high=np.float64(6.643974098104252))
('avg_rss12', 'iqr_3'): ConfidenceInterval(low=np.float64(4.335997250101495), high=np.float64(5.836564864076696))
('var_rss12', 'min'): ConfidenceInterval(low=np.float64(0.0), high=np.float64(0.0))
('var_rss12', 'max'): ConfidenceInterval(low=np.float64(4.623745569214122), high=np.float64(5.408399515717551))
('var_r

#### iv. Select Features

The three most important time-domain features for this dataset are:
* **min**: Captures the lowest value across the time series. This can help identify the lower limits of movement or sensor measurements and distinguish activities that produce different minimum readings.
* **max**: Captures the highest value across the time series. Together with the minimum, it provides the full range of observed values, which may help distinguish activities involving greater movement from those involving less movement.
* **mean**: Captures the typical value across the time series. This provides a measure of the overall level of sensor measurements and may help distinguish activities with consistently higher or lower readings.

Another important time-domain feature worth noting is the **standard deviation**. This is because it captures how much the sensor readings fluctuate throughout an activity from the mean, whereas the minimum and maximum tell us the full range of the values recorded. Unlike the minimum and maximum, which capture only the two extreme observations, standard deviation considers the variability of all observations. This makes this statistic useful for this particular dataset, especially to distinguish stationary activities (sitting, standing) from more active activities (walking, cycling).

## 2. ISLR 3.7.4

### (a) Linear Train


We would expect the training residual sum of squares (RSS) for the cubic regression to be lower than that of the linear regression, even if the true relationship between $X$ and $Y$ is linear. As we know, the RSS measures the sum of squared differences between actual $y_i$ and predicted $\hat{y}_i$ values. If we train the model using a cubic regression equation (which results in a more complex model fit to the data), then we expect this more complex model to fit the training data at least as well as a less complex model, such as the linear model. That is, the RSS will be less than or equal to that of the linear regression model, even if the true relationship between $X$ and $Y$ is linear, due to the additional flexibility of the cubic model.

Another version of this explanation: It should also be noted that the linear model is technically a special case of the cubic regression model (specifically, if $\beta_2 = 0$ and $\beta_3 = 0$). Because of this, the cubic model can always mimic the linear model if that happens to minimize RSS. By looking at it this way, we would still expect the training RSS to be less than or equal to the linear regression's RSS.

### (b) Linear Test

With regard to using test rather than training RSS: We would expect the residual sum of squares (RSS) for the cubic regression to be larger than that of the linear regression if the true relationship between $X$ and $Y$ is linear. As we know, the RSS measures the sum of squared residuals between actual $y_i$ and predicted $\hat{y}_i$ values. If the true relationship is linear, the linear model is expected to result in a lower test RSS because it captures the true pattern without reacting to the specific, random noise of the original 100 data points. If we train the model using a cubic regression equation (which allows for a more complex, curved fit than a linear model), and the true relationship is actually linear, then the cubic model may overfit the random noise in the training data. As a result, the squared differences between actual $y_i$ and predicted $\hat{y}_i$ values will, on average, be greater on the test data than those of the linear regression model.

### (c) Not Linear Train

If the true relationship between $X$ and $Y$ is not linear, we would expect the training RSS for the cubic regression to be lower than or equal to that of the linear regression. The logic is the same as the previous training question (with a true linear relationship): If we train the model using a cubic regression equation (which results in a more complex model fit to the data), then we expect this more complex model to fit the training data at least as well as a less complex model would, such as the linear model. Additionally, because the linear regression is a special case of the cubic regression (when $\beta_2 = 0$ and $\beta_3 = 0$), it still remains true that the cubic regression's training RSS can never be higher than the linear regression's training RSS.

### (d) Not Linear Testing

If the true relationship between $X$ and $Y$ is not linear, and we don't know how far it is from linear, for test data, we would not have enough information to tell whether the RSS would be higher or lower for the cubic regression compared to the linear regression. If the true relationship is slightly nonlinear, then the linear regression is expected to perform better because the cubic regression might be overfitting the data. On the other hand, if the true relationship is very nonlinear, the cubic regression might perform better. Thus, it could go both ways, and we don't have enough information to tell whose RSS would be lower in this scenario.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 